05 - subservice organizations & CUEC extraction + bridge letter parser

goal:
extract critical vendor risk governance items from soc2 reports:
1. sUBservice organizations: name, services provided, reporting method(carve-out vs inclusive) and listed complementary subservice organisation controls(CSOCs)
- flag carve-out subservice providers as high risk items requiring independent SOC2 reviews
2. complementary user entity controls(CUECs): identifier, full control requirement text and domain category
3. bridge letter parser: extract covered period, gap dates management statements of no material change, authorized signer and date



In [1]:
# CONFIG CELL - Execution parameters and directory paths
import os
import sys
from pathlib import Path

# Dynamically locate project root
CWD = Path(os.getcwd()).resolve()
if (CWD / "data").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "socrr" / "data").exists():
    PROJECT_ROOT = CWD / "socrr"
else:
    PROJECT_ROOT = CWD

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_LABELED_DIR = PROJECT_ROOT / "data" / "labeled"
INTERNAL_CONTROLS_DIR = PROJECT_ROOT / "data" / "internal_controls"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for d in [DATA_INTERIM_DIR, DATA_LABELED_DIR, INTERNAL_CONTROLS_DIR, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] Project Root: {PROJECT_ROOT}")
print(f"[CONFIG] Interim Data: {DATA_INTERIM_DIR}")
print(f"[CONFIG] Outputs Dir:  {OUTPUTS_DIR}")

[CONFIG] Project Root: D:\System and Organization controls 2 report info extraction
[CONFIG] Interim Data: D:\System and Organization controls 2 report info extraction\data\interim
[CONFIG] Outputs Dir:  D:\System and Organization controls 2 report info extraction\outputs


step 1 

In [2]:
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("SOCRR_Subservices")

KNOWN_SUBSERVICE_ORGS = [
    "Amazon Web Services", "AWS", "Microsoft Azure", "Azure", "Google Cloud Platform", "GCP",
    "Cloudflare", "Snowflake", "Datadog", "Salesforce", "Fastly", "Akamai", "MongoDB", "Twilio"
]

CUEC_CATEGORIES = {
    "access_control": [r"mfa", r"password", r"credential", r"access", r"deprovision", r"terminate", r"least privilege"],
    "endpoint_security": [r"firewall", r"antivirus", r"endpoint", r"patch", r"workstation", r"device"],
    "incident_management": [r"incident", r"notify", r"breach", r"alert", r"compromis"],
    "backup_recovery": [r"backup", r"retention", r"recovery", r"archive"],
    "data_protection": [r"encrypt", r"classif", r"privacy", r"gdpr", r"sensitive data", r"confidential"]
}

c:\Users\HP\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


step 2 - subservice and CUEC extractors

In [3]:
def extract_subservices_from_report(sec3_text: str, tables: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Extract subservice providers, reporting method, services, CSOCs, and risk flags."""
    subservices = []
    
    # 1. First look in tables for subservice disclosures
    for tbl in tables:
        rows = tbl["rows"]
        if not rows:
            continue
        first_row_str = " ".join(rows[0]).lower()
        if "subservice" in first_row_str or "sub-service" in first_row_str:
            # Table header mapping
            for row in rows[1:]:
                if len(row) >= 2 and any(row):
                    name = row[0].strip()
                    services = row[1].strip() if len(row) > 1 else ""
                    method = row[2].strip() if len(row) > 2 and ("carve" in row[2].lower() or "inclusive" in row[2].lower()) else "Carve-Out"
                    csocs = [row[3].strip()] if len(row) > 3 and row[3].strip() else []
                    
                    is_carve = "carve" in method.lower() or "carve" in sec3_text.lower()
                    subservices.append({
                        "name": name,
                        "method": "Carve-Out" if is_carve else "Inclusive",
                        "services": services,
                        "csocs": csocs,
                        "risk_flag": is_carve
                    })
                    
    # 2. Text heuristics fallback if no table was found
    if not subservices:
        for org in KNOWN_SUBSERVICE_ORGS:
            if re.search(rf"\b{re.escape(org)}\b", sec3_text):
                is_carve = bool(re.search(r"carve[\s-]?out", sec3_text, re.IGNORECASE))
                services_match = re.search(rf"{re.escape(org)}[\s\S]{{10,200}}?(?:for|provides?|services?)\s+([^.;\n]+)", sec3_text, re.IGNORECASE)
                services = services_match.group(1).strip() if services_match else "Data center colocation / Cloud infrastructure services"
                
                subservices.append({
                    "name": org,
                    "method": "Carve-Out" if is_carve else "Inclusive",
                    "services": services,
                    "csocs": ["Physical perimeter security, hardware maintenance, and uninterrupted power."],
                    "risk_flag": is_carve
                })
                
    # Deduplicate by name
    seen = set()
    deduped = []
    for s in subservices:
        if s["name"] not in seen and len(s["name"]) > 2:
            seen.add(s["name"])
            deduped.append(s)
            
    return deduped

def extract_cuecs_from_report(sec3_text: str, tables: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Extract Complementary User Entity Controls (CUECs) and categorize them."""
    cuecs = []
    seen_texts = set()
    
    # 1. Search tables for CUEC rows with robust multi-column inspection
    for tbl in tables:
        for row in tbl["rows"]:
            non_empty = [c.strip() for c in row if c and isinstance(c, str) and c.strip()]
            for cell in non_empty:
                # Target cells containing user entity responsibility statement
                if re.search(r"User\s+entities\s+are\s+responsible", cell, re.IGNORECASE) or (len(cell) > 35 and "user entit" in cell.lower()):
                    clean_text = cell.replace("\n", " ").strip()
                    if clean_text in seen_texts:
                        continue
                    seen_texts.add(clean_text)
                    
                    # Detect if a criteria code (e.g. CC6.1, CC7.3) exists in the row
                    crit = ""
                    for other in non_empty:
                        if other != cell and re.search(r"^(?:CC|A|C|PI|P)\d", other.strip()):
                            crit = other.strip().split()[0]
                            break
                            
                    cid = f"CUEC-{crit}" if crit else f"CUEC-{len(cuecs)+1:02d}"
                    cat = "other"
                    for c_name, patterns in CUEC_CATEGORIES.items():
                        if any(re.search(p, clean_text.lower()) for p in patterns):
                            cat = c_name
                            break
                            
                    cuecs.append({
                        "id": cid,
                        "text": clean_text,
                        "category": cat
                    })
                    
    # 2. Text fallback: parse numbered/bulleted CUECs in Section III
    if not cuecs:
        cuec_section_match = re.search(r"(?:COMPLEMENTARY\s+USER\s+ENTITY\s+CONTROLS|CUEC)[\s\S]{50,5000}", sec3_text, re.IGNORECASE)
        if cuec_section_match:
            c_text = cuec_section_match.group(0)
            items = re.findall(r"(?:^|\n)(?:([A-Z0-9.-]+)[:.]?\s+)?(User\s+entities\s+are\s+responsible[\s\S]+?)(?=\n[A-Z0-9.-]+[:.]|\n\n|COMPLEMENTARY|$)", c_text, re.IGNORECASE)
            for idx, item in enumerate(items):
                cid = item[0].strip() if item[0] else f"CUEC-{idx+1:02d}"
                text_content = item[1].replace("\n", " ").strip()
                if text_content in seen_texts:
                    continue
                seen_texts.add(text_content)
                cat = "other"
                for c_name, patterns in CUEC_CATEGORIES.items():
                    if any(re.search(p, text_content.lower()) for p in patterns):
                        cat = c_name
                        break
                cuecs.append({
                    "id": cid,
                    "text": text_content,
                    "category": cat
                })
                
    return cuecs

def parse_bridge_letter(text: str) -> Dict[str, Any]:
    """Extract gap period, no-material-change assertion, signer, and date from a bridge letter."""
    # Gap period
    gap_match = re.search(r"(?:period|throughout\s+the\s+period\s+from)?\s*([A-Za-z]+\s+[0-9]{1,2},\s+[0-9]{4}|[0-9]{1,2}\s+[A-Za-z]+\s+[0-9]{4})\s+(?:through|to)\s+([A-Za-z]+\s+[0-9]{1,2},\s+[0-9]{4}|[0-9]{1,2}\s+[A-Za-z]+\s+[0-9]{4})", text, re.IGNORECASE)
    start_dt = gap_match.group(1).strip() if gap_match else "N/A"
    end_dt = gap_match.group(2).strip() if gap_match else "N/A"
    
    # Material change statement
    no_change = bool(re.search(r"no\s+material\s+changes?|no\s+significant\s+changes?", text, re.IGNORECASE))
    
    # Signer and Date
    signer_match = re.search(r"(?:Sincerely|Regards|By)[:\s,\n]+([A-Za-z\s.]+)(?:\n([A-Za-z\s]+(?:CISO|Officer|Director|VP|Manager)))?", text, re.IGNORECASE)
    signer_name = signer_match.group(1).strip() if signer_match else "Authorized Representative"
    signer_title = signer_match.group(2).strip() if (signer_match and signer_match.group(2)) else "Executive Officer"
    
    date_match = re.search(r"(?:Date:?\s*)([A-Za-z]+\s+[0-9]{1,2},\s+[0-9]{4}|[0-9]{1,2}\s+[A-Za-z]+\s+[0-9]{4})", text, re.IGNORECASE)
    letter_date = date_match.group(1).strip() if date_match else "N/A"
    
    return {
        "is_bridge_letter": True,
        "period_start": start_dt,
        "period_end": end_dt,
        "statement_no_material_changes": no_change,
        "signer_name": signer_name,
        "signer_title": signer_title,
        "date": letter_date
    }

step 3 - run subservice, CUEC and bridge letter extactions

loop over all interim report files and save the extracted data structures

In [4]:
section_files = list(DATA_INTERIM_DIR.glob("*_sections.json"))
print(f"Processing subservices and CUECs across {len(section_files)} reports...")

all_subservices = []
all_cuecs = []
bridge_letters_data = []

for sf in section_files:
    report_id = sf.name.replace("_sections.json", "")
    
    # Check if this is a bridge letter
    if "bridge" in report_id.lower():
        pages_file = DATA_INTERIM_DIR / f"{report_id}_pages.json"
        if pages_file.exists():
            with open(pages_file, "r", encoding="utf-8") as pf:
                b_pages = json.load(pf)
                b_text = "\n".join(p["text"] for p in b_pages)
                b_res = parse_bridge_letter(b_text)
                b_res["report_id"] = report_id
                bridge_letters_data.append(b_res)
        continue
        
    with open(sf, "r", encoding="utf-8") as f:
        sections = json.load(f)
        
    tables_file = DATA_INTERIM_DIR / f"{report_id}_tables.json"
    tables = []
    if tables_file.exists():
        with open(tables_file, "r", encoding="utf-8") as tf:
            tables = json.load(tf)
            
    sec3_text = sections.get("section_3_system_description", {}).get("text", "")
    
    subservs = extract_subservices_from_report(sec3_text, tables)
    cuecs = extract_cuecs_from_report(sec3_text, tables)
    
    out_payload = {
        "report_id": report_id,
        "subservice_organizations": subservs,
        "cuecs": cuecs
    }
    
    out_path = DATA_INTERIM_DIR / f"{report_id}_subservice_cuec.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out_payload, f, indent=2, ensure_ascii=False)
        
    print(f"[{report_id}] Found {len(subservs)} subservice orgs and {len(cuecs)} CUECs.")
    for s in subservs:
        all_subservices.append({**s, "report_id": report_id})
    for c in cuecs:
        all_cuecs.append({**c, "report_id": report_id})

# Save bridge letters summary
with open(DATA_INTERIM_DIR / "bridge_letters.json", "w", encoding="utf-8") as f:
    json.dump(bridge_letters_data, f, indent=2)

# Export Excel summary
excel_path = OUTPUTS_DIR / "subservice_and_cuec_summary.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    if all_subservices:
        pd.DataFrame(all_subservices).to_excel(writer, sheet_name="Subservice Organizations", index=False)
    if all_cuecs:
        pd.DataFrame(all_cuecs).to_excel(writer, sheet_name="CUECs", index=False)
    if bridge_letters_data:
        pd.DataFrame(bridge_letters_data).to_excel(writer, sheet_name="Bridge Letters", index=False)

print(f"\nSummary workbook generated: {excel_path}")
if all_subservices:
    display(pd.DataFrame(all_subservices))
if all_cuecs:
    display(pd.DataFrame(all_cuecs).head(10))

Processing subservices and CUECs across 3 reports...
[sample_soc_report] Found 5 subservice orgs and 13 CUECs.
[synthetic_acme_soc2] Found 2 subservice orgs and 7 CUECs.

Summary workbook generated: D:\System and Organization controls 2 report info extraction\outputs\subservice_and_cuec_summary.xlsx


,name,method,services,csocs,risk_flag,report_id
0,CC6.1\nCC6.2\nCC6.3,Carve-Out,,[],True,sample_soc_report
1,CC6.4,Carve-Out,,[],True,sample_soc_report
2,CC7.1\nCC7.2\nCC7.3,Carve-Out,,[],True,sample_soc_report
3,CC7.2\nA1.2,Carve-Out,,[],True,sample_soc_report
4,CC8.1,Carve-Out,,[],True,sample_soc_report
5,Amazon Web Services\n(AWS),Carve-Out,"Physical data center security,\nredundant powe...","[AWS maintains physical perimeter security, bi...",True,synthetic_acme_soc2
6,Snowflake Computing,Carve-Out,Cloud data warehouse services for\naggregated ...,[Snowflake ensures encrypted volume management...,True,synthetic_acme_soc2


,id,text,category,report_id
0,CUEC-CC2.1,User entities are responsible for identifying ...,other,sample_soc_report
1,CUEC-02,User entities are responsible for the security...,data_protection,sample_soc_report
2,CUEC-CC2.3,User entities are responsible for assessing an...,other,sample_soc_report
3,CUEC-CC6.1,User entities are responsible for configuring ...,other,sample_soc_report
4,CUEC-05,User entities are responsible for changing the...,access_control,sample_soc_report
5,CUEC-06,User entities are responsible for the safeguar...,access_control,sample_soc_report
6,CUEC-CC6.6,"User entities are responsible for security, in...",data_protection,sample_soc_report
7,CUEC-CC6.6,"User entities are responsible for security, in...",data_protection,sample_soc_report
8,CUEC-CC6.2,User entities are responsible for managing acc...,access_control,sample_soc_report
9,CUEC-CC6.3,"User entities are responsible for requesting, ...",other,sample_soc_report


step 4 - sanity check

In [5]:
assert len(all_subservices) > 0, "No subservice organizations extracted!"
assert len(all_cuecs) > 0, "No CUECs extracted!"
assert (OUTPUTS_DIR / "subservice_and_cuec_summary.xlsx").exists()

print("Sanity Check Passed: Subservice organizations, CUECs, and Bridge letters successfully extracted.")

Sanity Check Passed: Subservice organizations, CUECs, and Bridge letters successfully extracted.
